In [7]:
from molearn.models.CNN_autoencoder import AutoEncoder as ConvolutionalAE
from molearn.models.foldingnet import AutoEncoder as FoldingNet
from molearn.trainers import OpenMM_Physics_Trainer
from molearn.data import PDBData
import datetime
import time
import os
import sys
import math
import torch

In [8]:
current_dir = os.path.dirname(os.path.abspath("notebook.ipynb"))
root_dir = os.path.join(current_dir, '..', '..' )
sys.path.append(root_dir)
from generic_utils.utils import AUTOENCODER_SELLECTION, AUTOENCODER_DEFAULT_MANDATORY_ARGUMENTS, get_data
from generic_utils.cli_utils import parse_all_args

In [9]:
data_path = "/home/alexandros/root/phd/data/aarhus/"
trajectories = [
    "stripped_inwards_open_1.xtc", "stripped_inwards_open_1.xtc", "stripped_inwards_open_1.xtc",
    "stripped_outwards_open_1.xtc", "stripped_outwards_open_1.xtc", "stripped_outwards_open_1.xtc"
]
topology = data_path + "stripped_inwards_open.pdb" # does it matter?
datafiles = [data_path + t for t in trajectories]
output_dir = '.'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder_of_choice = ConvolutionalAE

# Experiment specific
# model_kwargs = AUTOENCODER_DEFAULT_MANDATORY_ARGUMENTS["fold_net"] # TODO: THIS IS PROBLEMATIC
physics_weight = 0.0
patience = 32
batch_size = 32


In [10]:
data = get_data(datafiles=datafiles, topology=topology, atoms_select=True, fix_terminal=True)

Loading data...
Computed mean: 87.26246063590212, std: 23.109312562317257
Dataset shape: torch.Size([3006, 5474, 3])
Data loaded.


In [11]:
data.write_statistics(f"{output_dir}/data_statistics.json") # Save mean and std for analysis later

trainer = OpenMM_Physics_Trainer(device=device, physics_inter_weight=physics_weight)
trainer.set_data(data, 
                batch_size=batch_size, 
                validation_split=0.1, 
                manual_seed=25,
                save_indices=True,
                indices_dir=f"{output_dir}/indices"
                    )
trainer.prepare_physics(remove_NB=True)
trainer.set_autoencoder(autoencoder_of_choice)
trainer.prepare_optimiser()

device: cpu
using soft nonbonded forces by default
<Residue 0 (ILE) of chain 0>, 0, is a being set as a N terminal residue
<Residue 1107 (LEU) of chain 0> is a being set as a C terminal residue
nothing else
12753256.732768511


In [ ]:
fit_results = trainer.run_until_converge(
    patience=patience,
    log_filename="log.dat",
    log_folder=f"{output_dir}/logs",
    checkpoint_folder=f"{output_dir}/checkpoints",
    verbose=True,
)
print(fit_results)
print("Script complete. Exiting.")